# AEMO Price and Demand Data Extraction

This notebook generates AEMO Price and Demand CSV URLs, downloads the files, validates them, and merges them into a single dataset.

The code is organized into reusable functions so the extraction process is easier to maintain, reproduce, and extend.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import logging
import time

import pandas as pd
import requests


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

LOGGER = logging.getLogger(__name__)

## Configuration

Update the years, months, regions, and paths below depending on the extraction period you need.

In [ ]:
@dataclass(frozen=True)
class AEMOConfig:
    base_url: str = "https://aemo.com.au/aemo/data/nem/priceanddemand/PRICE_AND_DEMAND_"
    regions: tuple[str, ...] = ("NSW1", "TAS1", "QLD1", "VIC1", "SA1")
    months: tuple[str, ...] = ("01", "02", "03", "04")
    years: tuple[str, ...] = ("2025",)
    output_dir: Path = Path("data/raw/aemo_price_demand")
    url_manifest_path: Path = Path("data/aemo_urls.csv")
    merged_output_path: Path = Path("data/processed/aemo_price_demand_merged.csv")
    request_delay_seconds: float = 2.0
    request_timeout_seconds: int = 30


config = AEMOConfig()

## URL generation

In [ ]:
def build_aemo_url(base_url: str, year: str, month: str, region: str) -> str:
    """Build one AEMO Price and Demand CSV URL."""
    return f"{base_url}{year}{month}_{region}.csv"


def generate_aemo_urls(config: AEMOConfig) -> pd.DataFrame:
    """Generate the full URL manifest from configured years, months, and regions."""
    records = [
        {
            "year": year,
            "month": month,
            "region": region,
            "url": build_aemo_url(config.base_url, year, month, region),
        }
        for year in config.years
        for month in config.months
        for region in config.regions
    ]

    return pd.DataFrame(records)


def save_url_manifest(urls_df: pd.DataFrame, output_path: Path) -> None:
    """Save the URL manifest to disk."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    urls_df.to_csv(output_path, index=False)
    LOGGER.info("Saved URL manifest to %s with %d URLs.", output_path, len(urls_df))


urls_df = generate_aemo_urls(config)
save_url_manifest(urls_df, config.url_manifest_path)
urls_df.head()

## Manifest validation

In [ ]:
def inspect_url_manifest(urls_df: pd.DataFrame) -> None:
    """Print basic checks for the generated URL manifest."""
    required_columns = {"year", "month", "region", "url"}
    missing_columns = required_columns - set(urls_df.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    print("Basic information:")
    print(urls_df.info())

    print("\nFirst 5 rows:")
    display(urls_df.head())

    print("\nLast 5 rows:")
    display(urls_df.tail())

    print("\nMissing values:")
    print(urls_df.isna().sum())

    print("\nDuplicate rows:")
    print(urls_df.duplicated().sum())

    print("\nUnique years, months, and regions:")
    print(f"Years: {sorted(urls_df['year'].unique())}")
    print(f"Months: {sorted(urls_df['month'].unique())}")
    print(f"Regions: {sorted(urls_df['region'].unique())}")


inspect_url_manifest(urls_df)

## Download files

The downloader skips files that already exist, logs failed downloads, and uses a small delay between requests.

In [ ]:
REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/114.0.5735.199 Safari/537.36"
    ),
    "Referer": "https://aemo.com.au",
}


def download_file(
    url: str,
    save_path: Path,
    headers: dict[str, str] | None = None,
    timeout: int = 30,
    overwrite: bool = False,
) -> bool:
    """Download one file.

    Returns True when the file exists after the function completes, otherwise False.
    """
    save_path.parent.mkdir(parents=True, exist_ok=True)

    if save_path.exists() and not overwrite:
        LOGGER.info("Skipped existing file: %s", save_path)
        return True

    try:
        with requests.get(url, headers=headers, stream=True, timeout=timeout) as response:
            response.raise_for_status()

            with save_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file.write(chunk)

        LOGGER.info("Downloaded: %s", save_path)
        return True

    except requests.RequestException as exc:
        LOGGER.error("Failed to download %s: %s", url, exc)
        return False


def download_aemo_files(
    urls_df: pd.DataFrame,
    output_dir: Path,
    delay_seconds: float = 2.0,
    timeout_seconds: int = 30,
    overwrite: bool = False,
) -> pd.DataFrame:
    """Download all files listed in the URL manifest and return a download report."""
    report = []

    for row in urls_df.itertuples(index=False):
        file_name = Path(row.url).name
        save_path = output_dir / file_name

        success = download_file(
            url=row.url,
            save_path=save_path,
            headers=REQUEST_HEADERS,
            timeout=timeout_seconds,
            overwrite=overwrite,
        )

        report.append(
            {
                "year": row.year,
                "month": row.month,
                "region": row.region,
                "url": row.url,
                "file_path": str(save_path),
                "success": success,
            }
        )

        time.sleep(delay_seconds)

    return pd.DataFrame(report)


download_report = download_aemo_files(
    urls_df=urls_df,
    output_dir=config.output_dir,
    delay_seconds=config.request_delay_seconds,
    timeout_seconds=config.request_timeout_seconds,
    overwrite=False,
)

download_report.head()

## Merge downloaded CSV files

In [ ]:
EXPECTED_COLUMNS = {"REGION", "SETTLEMENTDATE", "TOTALDEMAND", "RRP", "PERIODTYPE"}


def read_aemo_csv(csv_path: Path) -> pd.DataFrame:
    """Read and lightly validate one AEMO CSV file."""
    df = pd.read_csv(csv_path)

    missing_columns = EXPECTED_COLUMNS - set(df.columns)
    if missing_columns:
        LOGGER.warning(
            "File %s is missing expected columns: %s",
            csv_path,
            sorted(missing_columns),
        )

    df["source_file"] = csv_path.name
    return df


def merge_csv_files(input_dir: Path, output_path: Path) -> pd.DataFrame:
    """Merge all downloaded CSV files into one processed dataset."""
    csv_files = sorted(input_dir.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {input_dir}")

    dataframes = []
    for csv_path in csv_files:
        try:
            dataframes.append(read_aemo_csv(csv_path))
            LOGGER.info("Loaded %s", csv_path)
        except Exception as exc:
            LOGGER.error("Failed to load %s: %s", csv_path, exc)

    if not dataframes:
        raise RuntimeError("No valid CSV files were loaded.")

    merged_df = pd.concat(dataframes, ignore_index=True)

    if {"REGION", "SETTLEMENTDATE"}.issubset(merged_df.columns):
        merged_df = merged_df.drop_duplicates(subset=["REGION", "SETTLEMENTDATE"])
        merged_df["SETTLEMENTDATE"] = pd.to_datetime(merged_df["SETTLEMENTDATE"], errors="coerce")
        merged_df = merged_df.sort_values(["REGION", "SETTLEMENTDATE"]).reset_index(drop=True)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    merged_df.to_csv(output_path, index=False)

    LOGGER.info("Saved merged dataset to %s with %d rows.", output_path, len(merged_df))
    return merged_df


merged_df = merge_csv_files(config.output_dir, config.merged_output_path)
merged_df.head()

## Summary checks

In [ ]:
def summarize_merged_data(df: pd.DataFrame) -> None:
    """Display basic quality checks for the merged dataset."""
    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nMissing values:")
    print(df.isna().sum())

    if "REGION" in df.columns:
        print("\nRows by region:")
        print(df["REGION"].value_counts().sort_index())

    if "SETTLEMENTDATE" in df.columns:
        print("\nDate range:")
        print(df["SETTLEMENTDATE"].min(), "to", df["SETTLEMENTDATE"].max())

    if {"REGION", "SETTLEMENTDATE"}.issubset(df.columns):
        print("\nDuplicate REGION + SETTLEMENTDATE pairs:")
        print(df.duplicated(subset=["REGION", "SETTLEMENTDATE"]).sum())


summarize_merged_data(merged_df)

## Optional: one-command pipeline

Use this cell when you want to rerun the full pipeline from URL generation to merged output.

In [ ]:
def run_pipeline(config: AEMOConfig, overwrite_downloads: bool = False) -> pd.DataFrame:
    """Run the complete AEMO extraction pipeline."""
    urls_df = generate_aemo_urls(config)
    save_url_manifest(urls_df, config.url_manifest_path)

    download_report = download_aemo_files(
        urls_df=urls_df,
        output_dir=config.output_dir,
        delay_seconds=config.request_delay_seconds,
        timeout_seconds=config.request_timeout_seconds,
        overwrite=overwrite_downloads,
    )

    failed_downloads = download_report.loc[~download_report["success"]]
    if not failed_downloads.empty:
        LOGGER.warning("%d downloads failed.", len(failed_downloads))
        display(failed_downloads)

    merged_df = merge_csv_files(config.output_dir, config.merged_output_path)
    summarize_merged_data(merged_df)

    return merged_df


# Uncomment to rerun the complete pipeline:
# merged_df = run_pipeline(config, overwrite_downloads=False)